In [76]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [52]:
import xgboost as xgb
import lightgbm as lgb
import shap

In [10]:
DATA_PATH = Path("data/creditcard.csv")
RESULTS_DIR = Path("results/")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
print(DATA_PATH.resolve(), "shape:", df_raw.shape)

C:\Users\lalas\Documents\job_prep\DataScience\practice_projects\fraud_detection\data\creditcard.csv shape: (284807, 31)


## Exploratory data analysis

In [11]:
eda_df = df_raw.copy()
print("Shape (rows, cols):", eda_df.shape)
print("Missing values (total):", int(eda_df.isna().sum().sum()))
if eda_df["Class"].nunique() == 2:
    vc = eda_df["Class"].value_counts(normalize=True)
    print("Class distribution:\n", vc)
    print(f"Fraud rate: {100 * float(vc.get(1, 0)):.4f}%")

Shape (rows, cols): (284807, 31)
Missing values (total): 0
Class distribution:
 Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64
Fraud rate: 0.1727%


In [12]:
t_span_sec = float(eda_df["Time"].max() - eda_df["Time"].min())
print(f"Time span: {t_span_sec / 3600:.2f} hours (~{t_span_sec / (3600 * 24):.2f} days)")
print("Time min / max:", eda_df["Time"].min(), eda_df["Time"].max())

Time span: 48.00 hours (~2.00 days)
Time min / max: 0.0 172792.0


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
eda_df["Amount"].hist(bins=80, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Amount")
axes[0].set_yscale("log")
np.log1p(eda_df["Amount"]).hist(bins=80, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("log1p(Amount)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "1_eda_amount_hist.png", dpi=120)
plt.close()

In [19]:
# Fraud rate over Time (binned)
eda_sorted = eda_df.sort_values("Time").reset_index(drop=True)
n_bins = 48
eda_sorted["time_bin"] = pd.qcut(
    eda_sorted["Time"].rank(method="first"),
    q=n_bins,
    duplicates="drop",
)
fraud_by_t = eda_sorted.groupby("time_bin", observed=True)["Class"].mean() * 100
plt.figure(figsize=(10, 4))
fraud_by_t.reset_index(drop=True).plot()
plt.xlabel("Time bin (ordered)")
plt.ylabel("% fraud (mean Class)")
plt.title("Fraud rate over time (quantile bins)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "2_eda_fraud_over_time.png", dpi=120)
plt.close()

In [20]:
# Correlation heatmap: sample of anonymized V columns + Amount (raw)
v_cols = [c for c in eda_df.columns if c.startswith("V")]
sample_vs = v_cols[:15] if len(v_cols) > 15 else v_cols
corr_cols = sample_vs + ["Amount"]
cm = eda_df[corr_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(cm, cmap="vlag", center=0.0, square=False)
plt.title("Correlation heatmap (sample of V features + Amount)")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "3_eda_corr_heatmap_sample.png", dpi=120)
plt.close()

In [21]:
# Multicollinearity proxy: max |r| off-diagonal on full V1–V28 (can be slow on full df)
v_only = [c for c in eda_df.columns if c.startswith("V")]
if v_only:
    corr_v = eda_df[v_only].corr().values
    np.fill_diagonal(corr_v, 0.0)
    max_abs_r = float(np.nanmax(np.abs(corr_v)))
    print(f"Max |correlation| among V-features (off-diagonal): {max_abs_r:.4f}")



Max |correlation| among V-features (off-diagonal): 0.0000


## Leakage-safe preprocessing

In [75]:
USE_TIME_FEATURES = True
ENABLE_TUNING = True
TUNING_ITERS = 12
TUNING_RANDOM_STATE = 42

In [23]:
def load_and_sort(path: Path = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df.sort_values("Time").reset_index(drop=True)

In [24]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["day"] = (out["Time"] // (3600 * 24)).astype(np.float64)
    hr_from_start = out["Time"] // 3600
    out["hr_of_day"] = (hr_from_start % 24).astype(np.float64)
    return out

In [25]:
def fit_amount_scaler(train_df: pd.DataFrame):
    """
    Fit a StandardScaler to log1p(Amount) on training data only.
    Returns the fitted scaler.
    """
    scaler = StandardScaler()
    train_amt = np.log1p(train_df[["Amount"]].values)
    scaler.fit(train_amt)
    return scaler

def transform_amount_with_scaler(df: pd.DataFrame, scaler: StandardScaler) -> pd.DataFrame:
    """
    Apply scaler to log1p(Amount) and add 'amt_log_std' column.
    Returns a new dataframe with the added column.
    """
    df = df.copy()
    df["amt_log_std"] = scaler.transform(np.log1p(df[["Amount"]].values)).ravel()
    return df

In [26]:
df = load_and_sort()

## 4. Time-base train/ validation/ test split

In [27]:
# Unique `Time` cutoffs: first ~50% of distinct times → train, next ~20% → validation, last ~30% → test.
# The scaler and any resampler are **never** fit on validation or test.

In [28]:
unique_times = np.sort(df["Time"].unique())
n_ut = len(unique_times)
val_start_time = unique_times[int(n_ut * 0.5)]
test_start_time = unique_times[-int(n_ut * 0.3)]

train_df = df[df["Time"] < val_start_time].copy()
val_df = df[(df["Time"] >= val_start_time) & (df["Time"] < test_start_time)].copy()
test_df = df[df["Time"] >= test_start_time].copy()

if USE_TIME_FEATURES:
    train_df = add_time_features(train_df)
    val_df = add_time_features(val_df)
    test_df = add_time_features(test_df)

In [29]:
scaler = fit_amount_scaler(train_df)
train_df = transform_amount_with_scaler(train_df, scaler)
val_df = transform_amount_with_scaler(val_df, scaler)
test_df = transform_amount_with_scaler(test_df, scaler)

In [30]:
exclude = {"Class", "Amount", "Time"} # since Time acts as dataset index proxy, it's not a business time feature
feature_cols = [c for c in train_df.columns if c not in exclude]

In [31]:
X_train = train_df[feature_cols].values
y_train = train_df["Class"].values.astype(np.int32)
X_val = val_df[feature_cols].values
y_val = val_df["Class"].values.astype(np.int32)
X_test = test_df[feature_cols].values
y_test = test_df["Class"].values.astype(np.int32)

In [32]:
# scale pos weight
spw = float((y_train == 0).sum()) / max(1, int((y_train == 1).sum()))
# smote k
n_minority = int((y_train == 1).sum())
smote_k = max(1, min(5, n_minority - 1))

In [33]:
print(
    "Split rows — train:",
    len(train_df),
    "val:",
    len(val_df),
    "test:",
    len(test_df),
)
print("Train fraud %:", 100 * float(train_df["Class"].mean()))
print("Val fraud %:", 100 * float(val_df["Class"].mean()))
print("Test fraud %:", 100 * float(test_df["Class"].mean()))
print("scale_pos_weight (neg/pos):", spw)
print("Feature count:", len(feature_cols))

Split rows — train: 144421 val: 48555 test: 91831
Train fraud %: 0.19387762167551809
Val fraud %: 0.2080115333127381
Test fraud %: 0.12087421458984438
scale_pos_weight (neg/pos): 514.7892857142857
Feature count: 31


## 5. Metrics and threshold tuning (validation only)

In [34]:
def scores_supervised(model, X: np.ndarray) -> np.ndarray:
    """
    Returns model scores for supervised binary classification.
    
    Uses `predict_proba` if available (preferred for probability), else falls back to raw `decision_function`
    output scaled to [0, 1]. The decision function is used for models that do not provide probabilities;
    it gives a continuous score reflecting distance from the decision boundary.

    Caveats:
    - Scaled decision function is not a true probability, just a relative confidence.
    - Scaling depends on min/max values in the dataset, which can vary between splits.
    - May require calibration for probability interpretation, especially on imbalanced data.
    """
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    dec = model.decision_function(X)
    lo, hi = dec.min(), dec.max()
    if hi <= lo:
        return np.full_like(dec, 0.5, dtype=float)
    return (dec - lo) / (hi - lo + 1e-12)

In [35]:
def tune_threshold_max_f1(y_true: np.ndarray, scores: np.ndarray, n_grid: int = 300):
    best_t, best_f1 = 0.5, -1.0
    lo, hi = float(scores.min()), float(scores.max())
    if hi <= lo:
        return 0.5, 0.0, 0.0, 0.0
    for t in np.linspace(lo, hi, n_grid):
        y_pred = (scores >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t
    y_pred = (scores >= best_t).astype(int)
    return (
        float(best_t),
        float(precision_score(y_true, y_pred, zero_division=0)),
        float(recall_score(y_true, y_pred, zero_division=0)),
        float(f1_score(y_true, y_pred, zero_division=0)),
    )


In [36]:
def val_metrics_ranking(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    pr = average_precision_score(y_true, scores)
    roc = roc_auc_score(y_true, scores)
    return float(pr), float(roc)

In [37]:
def test_metrics_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    return {
        "pr_auc": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

## 6. Models: fit on train, scores on val; store fitted objects

In [77]:
def tune_estimator_on_val(
    base_estimator,
    param_grid: dict,
    X_fit: np.ndarray,
    y_fit: np.ndarray,
    X_eval: np.ndarray,
    y_eval: np.ndarray,
    n_iter: int,
    random_state: int,
    model_name: str,
):
    """Random search on train only; rank candidates by validation PR-AUC."""
    if not ENABLE_TUNING or n_iter <= 0:
        est = clone(base_estimator)
        est.fit(X_fit, y_fit)
        return est, {}, float("nan")

    sampled_params = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=random_state))
    best_est = None
    best_params = {}
    best_pr_auc = -1.0
    for params in sampled_params:
        est = clone(base_estimator)
        est.set_params(**params)
        try:
            est.fit(X_fit, y_fit)
            s_eval = scores_supervised(est, X_eval)
            pr_auc = float(average_precision_score(y_eval, s_eval))
            if pr_auc > best_pr_auc:
                best_pr_auc = pr_auc
                best_params = params
                best_est = est
        except Exception as exc:
            print(f"  [tuning skip] {model_name} params={params} error={exc}")
    if best_est is None:
        est = clone(base_estimator)
        est.fit(X_fit, y_fit)
        return est, {}, float("nan")
    return best_est, best_params, best_pr_auc

In [38]:
val_rows: list[dict] = []
fitted: dict = {}

### Supervised Baselines

In [ ]:
# Baselines:** Logistic Regression, Random Forest, XGBoost, LightGBM with `class_weight` / `scale_pos_weight`
# and random-search hyperparameter tuning.

In [81]:
lr_base = LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced", solver="lbfgs")
lr_param_grid = {
    "C": np.logspace(-3, 2, 60),
}
lr, lr_best_params, lr_best_pr = tune_estimator_on_val(
    lr_base,
    lr_param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    n_iter=TUNING_ITERS,
    random_state=TUNING_RANDOM_STATE,
    model_name="LR_balanced",
)
if lr_best_params:
    print(f"  LR_balanced tuned params: {lr_best_params} (val PR-AUC={lr_best_pr:.6f})")

  LR_balanced tuned params: {'C': np.float64(0.0103979841848149)} (val PR-AUC=0.822896)


In [82]:

s_val = scores_supervised(lr, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "LR_balanced",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["LR_balanced"] = {"type": "sklearn", "estimator": lr, "threshold": thr}
print(val_rows[-1])

{'model': 'LR_balanced', 'val_pr_auc': 0.8228957435134737, 'val_roc_auc': 0.9743423077190287, 'val_precision': 0.581081081081081, 'val_recall': 0.8514851485148515, 'val_f1': 0.6907630522088354, 'threshold': 0.9966555183946488}


In [83]:
rf_base = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1,
    max_depth=12,
)
rf_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [8, 12, 16, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", 0.5, 0.8],
}
rf, rf_best_params, rf_best_pr = tune_estimator_on_val(
    rf_base,
    rf_param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    n_iter=TUNING_ITERS,
    random_state=TUNING_RANDOM_STATE + 1,
    model_name="RF_balanced_subsample",
)
if rf_best_params:
    print(f"  RF_balanced_subsample tuned params: {rf_best_params} (val PR-AUC={rf_best_pr:.6f})")

  RF_balanced_subsample tuned params: {'n_estimators': 200, 'min_samples_leaf': 8, 'max_features': 'sqrt', 'max_depth': 16} (val PR-AUC=0.797767)


In [84]:
s_val = scores_supervised(rf, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "RF_balanced_subsample",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["RF_balanced_subsample"] = {"type": "sklearn", "estimator": rf, "threshold": thr}
print(val_rows[-1])

{'model': 'RF_balanced_subsample', 'val_pr_auc': 0.7977667999755624, 'val_roc_auc': 0.9844892185177573, 'val_precision': 0.8602150537634409, 'val_recall': 0.7920792079207921, 'val_f1': 0.8247422680412371, 'threshold': 0.54899441078991}


In [85]:
xgb_base = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    scale_pos_weight=spw,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
)
xgb_param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [1.0, 3.0, 10.0],
    "scale_pos_weight": [spw * 0.5, spw, spw * 1.5],
}
xgb_clf, xgb_best_params, xgb_best_pr = tune_estimator_on_val(
    xgb_base,
    xgb_param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    n_iter=TUNING_ITERS,
    random_state=TUNING_RANDOM_STATE + 2,
    model_name="XGB_scale_pos_weight",
)
if xgb_best_params:
    print(f"  XGB_scale_pos_weight tuned params: {xgb_best_params} (val PR-AUC={xgb_best_pr:.6f})")

  XGB_scale_pos_weight tuned params: {'subsample': 0.7, 'scale_pos_weight': 257.39464285714286, 'reg_lambda': 10.0, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7} (val PR-AUC=0.813554)


In [86]:

s_val = scores_supervised(xgb_clf, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "XGB_scale_pos_weight",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["XGB_scale_pos_weight"] = {"type": "sklearn", "estimator": xgb_clf, "threshold": thr}
print(val_rows[-1])

{'model': 'XGB_scale_pos_weight', 'val_pr_auc': 0.8135544945437932, 'val_roc_auc': 0.9878807173242192, 'val_precision': 0.8651685393258427, 'val_recall': 0.7623762376237624, 'val_f1': 0.8105263157894737, 'threshold': 0.9830240686483225}


In [87]:
lgb_base = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    class_weight=None,
    scale_pos_weight=spw,
    n_jobs=-1,
    verbose=-1,
)
lgb_param_grid = {
    "n_estimators": [200, 400, 600],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "min_child_samples": [20, 50, 100],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
    "scale_pos_weight": [spw * 0.5, spw, spw * 1.5],
}
lgb_clf, lgb_best_params, lgb_best_pr = tune_estimator_on_val(
    lgb_base,
    lgb_param_grid,
    X_train,
    y_train,
    X_val,
    y_val,
    n_iter=TUNING_ITERS,
    random_state=TUNING_RANDOM_STATE + 3,
    model_name="LGBM_scale_pos_weight",
)
if lgb_best_params:
    print(f"  LGBM_scale_pos_weight tuned params: {lgb_best_params} (val PR-AUC={lgb_best_pr:.6f})")

  LGBM_scale_pos_weight tuned params: {'subsample': 0.85, 'scale_pos_weight': 514.7892857142857, 'reg_lambda': 5.0, 'num_leaves': 63, 'n_estimators': 400, 'min_child_samples': 100, 'learning_rate': 0.01, 'colsample_bytree': 1.0} (val PR-AUC=0.781326)


In [88]:

s_val = scores_supervised(lgb_clf, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "LGBM_scale_pos_weight",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["LGBM_scale_pos_weight"] = {"type": "sklearn", "estimator": lgb_clf, "threshold": thr}
print(val_rows[-1])

{'model': 'LGBM_scale_pos_weight', 'val_pr_auc': 0.7813262404093958, 'val_roc_auc': 0.9764733479993477, 'val_precision': 0.8494623655913979, 'val_recall': 0.7821782178217822, 'val_f1': 0.8144329896907216, 'threshold': 0.9455444702921707}


### SMOTE/ ADASYN

In [ ]:
# **Resampling:** `imblearn.pipeline.Pipeline` with SMOTE or ADASYN **only** before the classifier on training folds;
# `sampling_strategy` grid plus random-search tuning on classifier parameters.

In [89]:
def fit_eval_imblearn_pipeline(name: str, pipe: ImbPipeline, param_grid: dict | None = None) -> None:
    if ENABLE_TUNING and param_grid:
        candidate_params = list(
            ParameterSampler(param_grid, n_iter=TUNING_ITERS, random_state=TUNING_RANDOM_STATE + 10)
        )
        best_pipe = None
        best_params = {}
        best_pr_auc = -1.0
        for params in candidate_params:
            p = clone(pipe)
            p.set_params(**params)
            try:
                p.fit(X_train, y_train)
                clf = p.named_steps["clf"]
                s_val = clf.predict_proba(X_val)[:, 1]
                pr_auc = float(average_precision_score(y_val, s_val))
                if pr_auc > best_pr_auc:
                    best_pr_auc = pr_auc
                    best_params = params
                    best_pipe = p
            except Exception as e:
                print(f"Skipping {name} candidate params {params}: {e}")
        if best_pipe is not None:
            pipe = best_pipe
            print(f"  {name} tuned params: {best_params} (val PR-AUC={best_pr_auc:.6f})")
    try:
        pipe.fit(X_train, y_train)
    except ValueError as e:
        print(f"Skipping {name}: {e}")
        return
    clf = pipe.named_steps["clf"]
    s_val = clf.predict_proba(X_val)[:, 1]
    pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
    thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
    val_rows.append(
        {
            "model": name,
            "val_pr_auc": pr_auc,
            "val_roc_auc": roc_auc,
            "val_precision": p,
            "val_recall": r,
            "val_f1": f1v,
            "threshold": thr,
        }
    )
    fitted[name] = {"type": "imblearn", "pipeline": pipe, "threshold": thr}



In [90]:
# SMOTE grid: 5%, 10%, 50% minority-to-majority ratio after sampling (imblearn `sampling_strategy` float)
for strat in (0.05, 0.10, 0.50):
    pipe_lr = ImbPipeline(
        [
            ("smote", SMOTE(sampling_strategy=strat, random_state=42, k_neighbors=smote_k)),
            (
                "clf",
                LogisticRegression(max_iter=2000, random_state=42, solver="lbfgs"),
            ),
        ]
    )
    smote_lr_grid = {
        "smote__sampling_strategy": [0.05, 0.10, 0.20, 0.50],
        "smote__k_neighbors": [max(1, min(3, smote_k)), smote_k],
        "clf__C": np.logspace(-3, 2, 40),
    }
    fit_eval_imblearn_pipeline(f"SMOTE_{strat:.2f}_LR", pipe_lr, smote_lr_grid)

    print(val_rows[-1])

    pipe_rf = ImbPipeline(
        [
            ("smote", SMOTE(sampling_strategy=strat, random_state=42, k_neighbors=smote_k)),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=200,
                    random_state=42,
                    n_jobs=-1,
                    max_depth=12,
                ),
            ),
        ]
    )
    smote_rf_grid = {
        "smote__sampling_strategy": [0.05, 0.10, 0.20, 0.50],
        "smote__k_neighbors": [max(1, min(3, smote_k)), smote_k],
        "clf__n_estimators": [200, 300, 500],
        "clf__max_depth": [8, 12, 16, None],
        "clf__min_samples_leaf": [1, 2, 4],
    }
    fit_eval_imblearn_pipeline(f"SMOTE_{strat:.2f}_RF", pipe_rf, smote_rf_grid)

    print(val_rows[-1])


  SMOTE_0.05_LR tuned params: {'smote__sampling_strategy': 0.5, 'smote__k_neighbors': 5, 'clf__C': np.float64(0.014251026703029978)} (val PR-AUC=0.824671)
{'model': 'SMOTE_0.05_LR', 'val_pr_auc': 0.8246706557877306, 'val_roc_auc': 0.9729801910723124, 'val_precision': 0.7226890756302521, 'val_recall': 0.8514851485148515, 'val_f1': 0.7818181818181819, 'threshold': 0.996655518394649}
  SMOTE_0.05_RF tuned params: {'smote__sampling_strategy': 0.1, 'smote__k_neighbors': 3, 'clf__n_estimators': 200, 'clf__min_samples_leaf': 1, 'clf__max_depth': 8} (val PR-AUC=0.793152)
{'model': 'SMOTE_0.05_RF', 'val_pr_auc': 0.7931518592375667, 'val_roc_auc': 0.9924973241947962, 'val_precision': 0.8191489361702128, 'val_recall': 0.7623762376237624, 'val_f1': 0.7897435897435897, 'threshold': 0.6284460548835762}
  SMOTE_0.10_LR tuned params: {'smote__sampling_strategy': 0.5, 'smote__k_neighbors': 5, 'clf__C': np.float64(0.014251026703029978)} (val PR-AUC=0.824671)
{'model': 'SMOTE_0.10_LR', 'val_pr_auc': 0.82

In [91]:
# ADASYN (same grid; may fail if neighborhood geometry is degenerate)
ada_k = max(1, min(5, n_minority - 1))
for strat in (0.05, 0.10, 0.50):
    ada_pipe = ImbPipeline(
        [
            ("adasyn", ADASYN(sampling_strategy=strat, random_state=42, n_neighbors=ada_k)),
            ("clf", LogisticRegression(max_iter=2000, random_state=42, solver="lbfgs")),
        ]
    )
    adasyn_lr_grid = {
        "adasyn__sampling_strategy": [0.05, 0.10, 0.20, 0.50],
        "adasyn__n_neighbors": [max(1, min(3, ada_k)), ada_k],
        "clf__C": np.logspace(-3, 2, 40),
    }
    fit_eval_imblearn_pipeline(f"ADASYN_{strat:.2f}_LR", ada_pipe, adasyn_lr_grid)
    print(val_rows[-1])

  ADASYN_0.05_LR tuned params: {'clf__C': np.float64(0.15117750706156616), 'adasyn__sampling_strategy': 0.2, 'adasyn__n_neighbors': 5} (val PR-AUC=0.822469)
{'model': 'ADASYN_0.05_LR', 'val_pr_auc': 0.8224689459906709, 'val_roc_auc': 0.9641752696341166, 'val_precision': 0.6013986013986014, 'val_recall': 0.8514851485148515, 'val_f1': 0.7049180327868853, 'threshold': 0.9966555183946488}
  ADASYN_0.10_LR tuned params: {'clf__C': np.float64(0.15117750706156616), 'adasyn__sampling_strategy': 0.2, 'adasyn__n_neighbors': 5} (val PR-AUC=0.822469)
{'model': 'ADASYN_0.10_LR', 'val_pr_auc': 0.8224689459906709, 'val_roc_auc': 0.9641752696341166, 'val_precision': 0.6013986013986014, 'val_recall': 0.8514851485148515, 'val_f1': 0.7049180327868853, 'threshold': 0.9966555183946488}
  ADASYN_0.50_LR tuned params: {'clf__C': np.float64(0.15117750706156616), 'adasyn__sampling_strategy': 0.2, 'adasyn__n_neighbors': 5} (val PR-AUC=0.822469)
{'model': 'ADASYN_0.50_LR', 'val_pr_auc': 0.8224689459906709, 'val_

### Anomaly Detection

In [78]:
# **Anomaly:** Isolation Forest and One-Class SVM on normal-only training rows; higher anomaly score = more fraudulent.

In [46]:
def anomaly_scores_iforest(model: IsolationForest, X: np.ndarray) -> np.ndarray:
    return -model.score_samples(X)


def anomaly_scores_ocsvm(model: OneClassSVM, X: np.ndarray) -> np.ndarray:
    return -model.decision_function(X)


def subsample_normal(X: np.ndarray, y: np.ndarray, max_samples: int, random_state: int = 42):
    rng = np.random.RandomState(random_state)
    idx_normal = np.where(y == 0)[0]
    if len(idx_normal) <= max_samples:
        return X[idx_normal]
    chosen = rng.choice(idx_normal, size=max_samples, replace=False)
    return X[chosen]

In [47]:
# Isolation Forest (normal-only train)
Xn_if = subsample_normal(X_train, y_train, max_samples=50_000)
iforest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination=0.001,
    random_state=42,
    n_jobs=-1,
)
iforest.fit(Xn_if)
s_val = anomaly_scores_iforest(iforest, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "IForest_normal_only",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["IForest_normal_only"] = {"type": "anomaly_iforest", "estimator": iforest, "threshold": thr}
print(val_rows[-1])

{'model': 'IForest_normal_only', 'val_pr_auc': 0.32282632364125585, 'val_roc_auc': 0.973471215120026, 'val_precision': 0.4444444444444444, 'val_recall': 0.39603960396039606, 'val_f1': 0.418848167539267, 'threshold': 0.6678000543644834}


In [48]:

# One-Class SVM (normal-only train, subsampled for runtime)
Xn_svm = subsample_normal(X_train, y_train, max_samples=25_000)
ocsvm = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)
ocsvm.fit(Xn_svm)
s_val = anomaly_scores_ocsvm(ocsvm, X_val)
pr_auc, roc_auc = val_metrics_ranking(y_val, s_val)
thr, p, r, f1v = tune_threshold_max_f1(y_val, s_val)
val_rows.append(
    {
        "model": "OneClassSVM_normal_only",
        "val_pr_auc": pr_auc,
        "val_roc_auc": roc_auc,
        "val_precision": p,
        "val_recall": r,
        "val_f1": f1v,
        "threshold": thr,
    }
)
fitted["OneClassSVM_normal_only"] = {"type": "anomaly_ocsvm", "estimator": ocsvm, "threshold": thr}
print(val_rows[-1])

{'model': 'OneClassSVM_normal_only', 'val_pr_auc': 0.3676582993082381, 'val_roc_auc': 0.9757128022209081, 'val_precision': 0.6666666666666666, 'val_recall': 0.37623762376237624, 'val_f1': 0.4810126582278481, 'threshold': 423.9094131171733}


## 7. Validation comparison (PR-AUC primary)

In [49]:
val_table = pd.DataFrame(val_rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
print("\n=== Validation comparison (sorted by PR-AUC) ===\n")
print(val_table.to_string(index=False))
val_table.to_csv(RESULTS_DIR / "validation_comparison.csv", index=False)

best_name = str(val_table.iloc[0]["model"])
best_thr = float(val_table.iloc[0]["threshold"])
print(f"\nBest model by validation PR-AUC: {best_name}")



=== Validation comparison (sorted by PR-AUC) ===

                  model  val_pr_auc  val_roc_auc  val_precision  val_recall   val_f1  threshold
          SMOTE_0.05_LR    0.811213     0.972292       0.745455    0.811881 0.777251   0.816054
          SMOTE_0.10_LR    0.808464     0.971240       0.732143    0.811881 0.769953   0.933110
   XGB_scale_pos_weight    0.799232     0.988951       0.865169    0.762376 0.810526   0.983268
            LR_balanced    0.794136     0.954906       0.613139    0.831683 0.705882   0.996656
  RF_balanced_subsample    0.780227     0.986326       0.869565    0.792079 0.829016   0.420413
         ADASYN_0.05_LR    0.773882     0.963804       0.760000    0.752475 0.756219   0.983278
          SMOTE_0.50_LR    0.765886     0.946802       0.719626    0.762376 0.740385   0.996656
         ADASYN_0.50_LR    0.759111     0.938639       0.980000    0.485149 0.649007   1.000000
          SMOTE_0.10_RF    0.750524     0.987515       0.760000    0.752475 0.756219 

## 8. Final test — **once**, for the validation-selected model only

In [53]:
best_info = fitted[best_name]
if best_info["type"] == "sklearn":
    est = best_info["estimator"]
    s_test = scores_supervised(est, X_test)
elif best_info["type"] == "imblearn":
    clf = best_info["pipeline"].named_steps["clf"]
    s_test = clf.predict_proba(X_test)[:, 1]
elif best_info["type"] == "anomaly_iforest":
    s_test = anomaly_scores_iforest(best_info["estimator"], X_test)
elif best_info["type"] == "anomaly_ocsvm":
    s_test = anomaly_scores_ocsvm(best_info["estimator"], X_test)
else:
    raise RuntimeError("Unknown model type")

In [54]:
print(best_info)

{'type': 'imblearn', 'pipeline': Pipeline(steps=[('smote', SMOTE(random_state=42, sampling_strategy=0.05)),
                ('clf', LogisticRegression(max_iter=2000, random_state=42))]), 'threshold': 0.8160535117424754}


In [56]:

test_out = test_metrics_at_threshold(y_test, s_test, best_thr)
test_out["model"] = best_name
test_out["threshold_from_val"] = best_thr

print("\n=== Final test (single evaluation) ===\n")
for k, v in test_out.items():
    print(f"  {k}: {v}")

with open(RESULTS_DIR / "final_test_best_model.json", "w", encoding="utf-8") as f:
    json.dump(test_out, f, indent=2)



=== Final test (single evaluation) ===

  pr_auc: 0.6965211294243491
  roc_auc: 0.9617406874820743
  precision: 0.6384615384615384
  recall: 0.7477477477477478
  f1: 0.6887966804979253
  model: SMOTE_0.05_LR
  threshold_from_val: 0.8160535117424754


In [57]:
summary = {
    "data_path": str(DATA_PATH),
    "split": {
        "val_start_time": float(val_start_time),
        "test_start_time": float(test_start_time),
        "train_rows": int(len(train_df)),
        "val_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
    },
    "best_model_val_pr_auc": float(val_table.iloc[0]["val_pr_auc"]),
    "best_model_name": best_name,
    "final_test": test_out,
    "feature_columns": feature_cols,
}

In [58]:
summary

{'data_path': 'data\\creditcard.csv',
 'split': {'val_start_time': 86115.0,
  'test_start_time': 129939.0,
  'train_rows': 144421,
  'val_rows': 48555,
  'test_rows': 91831},
 'best_model_val_pr_auc': 0.8112133422423652,
 'best_model_name': 'SMOTE_0.05_LR',
 'final_test': {'pr_auc': 0.6965211294243491,
  'roc_auc': 0.9617406874820743,
  'precision': 0.6384615384615384,
  'recall': 0.7477477477477478,
  'f1': 0.6887966804979253,
  'model': 'SMOTE_0.05_LR',
  'threshold_from_val': 0.8160535117424754},
 'feature_columns': ['V1',
  'V2',
  'V3',
  'V4',
  'V5',
  'V6',
  'V7',
  'V8',
  'V9',
  'V10',
  'V11',
  'V12',
  'V13',
  'V14',
  'V15',
  'V16',
  'V17',
  'V18',
  'V19',
  'V20',
  'V21',
  'V22',
  'V23',
  'V24',
  'V25',
  'V26',
  'V27',
  'V28',
  'day',
  'hr_of_day',
  'amt_log_std']}

## Explainability

In [59]:
def pick_best_tree_for_shap(val_tbl: pd.DataFrame) -> str | None:
    """First validation row that uses an estimator with `feature_importances_` (tree booster)."""
    for _, row in val_tbl.iterrows():
        name = str(row["model"])
        if name not in fitted:
            continue
        info = fitted[name]
        if info["type"] == "sklearn":
            if hasattr(info["estimator"], "feature_importances_"):
                return name
        if info["type"] == "imblearn":
            clf = info["pipeline"].named_steps["clf"]
            if hasattr(clf, "feature_importances_"):
                return name
    return None

In [63]:

shap_model_name = pick_best_tree_for_shap(val_table) or best_name
shap_info = fitted.get(shap_model_name, fitted[best_name])
print(shap_model_name)
print(shap_info)

XGB_scale_pos_weight
{'type': 'sklearn', 'estimator': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...), 'threshold': 0.9832681077958324}


In [65]:
# Tree feature importances (RF / XGB / LGBM / RF inside SMOTE pipeline)
imp_series: pd.Series | None = None
if shap_info["type"] == "sklearn":
    est = shap_info["estimator"]
    if hasattr(est, "feature_importances_"):
        imp_series = pd.Series(est.feature_importances_, index=feature_cols).sort_values(ascending=False)
elif shap_info["type"] == "imblearn":
    clf = shap_info["pipeline"].named_steps["clf"]
    if hasattr(clf, "feature_importances_"):
        imp_series = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

if imp_series is not None:
    print(f"\nTop 15 feature importances ({shap_model_name}):\n")
    print(imp_series.head(15).to_string())
    imp_series.head(25).to_csv(RESULTS_DIR / "feature_importance_top.csv")
    plt.figure(figsize=(8, 6))
    imp_series.head(20).iloc[::-1].plot(kind="barh")
    plt.title(f"Feature importance — {shap_model_name}")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "4_feature_importance_bar.png", dpi=120)
    plt.close()


Top 15 feature importances (XGB_scale_pos_weight):

V14            0.425721
V10            0.133671
V12            0.047427
V4             0.036216
V8             0.031044
V20            0.025979
V7             0.022421
V3             0.021427
V21            0.019416
V19            0.017247
amt_log_std    0.016905
V13            0.016582
V17            0.016508
V23            0.015067
V1             0.014311
